# March Mania · Possession-accounting matchup features
**Milestone 08 — preserve negative evidence → engineer four candidates → control for basic rates → test all four seasons.**

The women’s temporal-change family failed its additional-season gate. We do not retune it or run its skipped component fits. This notebook tests a different representation: scoring opportunities retained after turnovers, extra attempts from rebounding, and free-throw production.

The 16-input reference stays fixed. Six ordinary rate controls are a **comparison group, not six novel discoveries**. Four nonlinear features must beat that rate-augmented comparison. This is an exploratory study on reused 2016–2019 seasons, not a 2026 leaderboard improvement.

Run the packaged tests first. Use **Python (March Mania)**. Keep the prior research folders. No network downloads, Git writes, environment installs, or AWS API actions occur.

**Resume note:** The checkpoint-metadata correction recomputes optional log loss from verified predictions. Previously completed snapshots and fits are migrated by checksum; features and training rules are unchanged.


In [ ]:
from pathlib import Path
import os, sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / 'run_round08.py').is_file():
    KIT = Path.home() / 'march_possession_research'
assert (KIT / 'run_round08.py').is_file(), 'Open the notebook from march_possession_research.'
sys.path.insert(0, str(KIT))
from run_round08 import run_stage
from possession_plots import figures
pio.renderers.default = 'plotly_mimetype'
print('Kernel:', sys.executable)
print('Research kit:', KIT)
print('Outputs stay outside the source repository.')

## 1. Preserve the actual replication result
The mean additional-season Brier change was **+0.0017966**. One of two additional seasons improved, so `STOP_EXPANSION` and `SKIPPED_BY_GATE` were correct. Do not rerun notebook 07 or change its gate.

These values are from the uploaded report, not new fits.

In [ ]:
prior = pd.read_csv(KIT / 'evidence/round07/replication_metrics.csv')
print('Uploaded temporal-change comparison:')
display(prior[['Gender','Season','recipe','brier','delta_vs_anchor','role']].round(7))
print('No prior experiment is rerun by this display.')

## 2. Build rate tables and verify all existing references
The snapshot functions see only pre-cutoff regular-season detailed games. They build same-season priors separately for men and women. Seeds remain in the unchanged reference, available after field announcement.

**Preparation:** reuse 14 base snapshots, create 14 small rate tables with no rating fits, and replay seven saved reference classifiers. Replay reads historical labels only to verify saved probabilities and Brier; those labels are never passed to the rate builder. A missing cache stops rather than triggering a rebuild.

Budget: 300 seconds, 15-second heartbeats.

In [ ]:
run_stage('prepare', max_seconds=300)
RUN = Path(json.loads((KIT / 'reports/latest_run.json').read_text())['run_dir'])
print(json.dumps(json.loads((RUN / 'prepare.json').read_text()), indent=2))
display(pd.read_csv(RUN / 'prior_replay.csv').round(7))

### Definitions and testable limitations
The six controls are offense/allowed turnover rate, offensive-rebound share, and FTA/FGA. Those concepts already exist in the wider project. The four new candidates are a neutral scoring proxy and its rebound, turnover, and free-throw accounting contrasts.

The scoring proxy follows the approximate identity:

$$\mathrm{PPP}=\frac{(1-t)[2e+af]}{1-r(1-g)+0.475f}$$

Here $t$ is turnovers per possession, $r$ is offensive-rebound share, $g$ is field-goal accuracy, $e$ is effective field-goal accuracy, $a$ is free-throw accuracy, and $f$ is FTA/FGA. Opposing profiles are combined using a fixed odds-based rule; FT accuracy uses offense only.

This is a deliberately simplified accounting hypothesis, **not** play-by-play simulation, a causal decomposition, or an already-calibrated forecast. Rebounds after free throws and team/dead-ball rebounds are not fully represented. See `RESEARCH_PLAN.md` for derivation, priors, domain sources, and limitations.

In [ ]:
registry = pd.read_csv(RUN / 'feature_registry.csv')
display(registry[['feature','family','new_definition','description']])
display(pd.read_csv(RUN / 'coverage.csv'))

## 3. Four-season controlled experiment
| Recipe | Inputs | Purpose |
|---|---:|---|
| Anchor | 16 | Unchanged reference |
| Anchor + basic rates | 22 | Stronger direct-rate comparison |
| Anchor + accounting | 20 | New representations alone |
| Anchor + both | 26 | Primary incremental-feature test |

Both populations; 2016, 2017, 2018, 2019. Training for each year uses 2013 through the preceding year. Main-draw labels only. Exactly 32 comparisons: seven upstream reference replays and at most **25 new classifiers**. The new classifier budget includes the missing men’s 2016 reference. There are no rating fits or hyperparameter/temperature/ensemble searches.

**Primary question:** does `anchor_both` beat `anchor_rates`? All comparisons keep other inputs fixed. Budget: 300 seconds; each completed fit is checkpointed.

In [ ]:
run_stage('evaluate', max_seconds=300)
metrics = pd.read_csv(RUN / 'metrics.csv')
display(metrics[['Gender','Season','recipe','brier','delta_vs_anchor','source']].round(7))
print(json.dumps(json.loads((RUN / 'evaluation_receipt.json').read_text()), indent=2))

## 4. Read conditional feature effects, not just a smallest score
Negative `delta_brier` favors adding the named family. `mechanism_given_rates` compares the nonlinear additions against the reference that already sees the basic rates.

A mean improvement of at least 0.0005, improvement in at least three of four seasons, and worst deterioration no larger than 0.003 triggers **consideration of later-era replication only**. It is not significance, production promotion, or an automatic next job. These historical seasons have been repeatedly used.

In [ ]:
ablations = pd.read_csv(RUN / 'ablations.csv')
display(ablations.query("comparison == 'mechanism_given_rates'").round(7))
display(pd.read_csv(RUN / 'aggregate.csv').round(7))
print(json.dumps(json.loads((RUN / 'decisions.json').read_text()), indent=2))

## 5. Interactive evidence
The charts cover the prior negative result, feature groups, data support, paired accounting profiles, all four-season scores, the primary comparison, fixed-family effects, calibration, and training-only overlap. A table of coefficients is retained in the return package; it is not causal feature importance.

In [ ]:
plots = figures(RUN, KIT / 'evidence/round07')
assert len(plots) == 10
for figure in plots[:5]:
    figure.show()

In [ ]:
for figure in plots[5:]:
    figure.show()
display(pd.read_csv(RUN / 'training_redundancy.csv').round(4))

## 6. Save and return the measured evidence
Report limit: 120 seconds. Raw files, source, and upstream cache preservation are checked before export. The ZIP excludes raw rows, model objects, private notebook edits, and per-game predictions.

Save this notebook with **Ctrl+S** and return `reports/milestone_08_return.zip`. Stop here. A technically complete experiment can still reject all hypotheses. No submission is created.

In [ ]:
run_stage('report', max_seconds=120)
report = json.loads((KIT / 'reports/latest_report.json').read_text())
print('Return archive:', report['return_zip'])
print('Interactive HTML:', report['html'])
display(FileLink(str(Path(report['return_zip']).relative_to(KIT))))
display(FileLink(str(Path(report['html']).relative_to(KIT))))